# Meridian — Financial Diffusion Augmentation (FDA) Generation (Colab)

Runs the FDA offline pipeline (§8.2–8.3 of `meridian-spec.md`):
1. Trains a regime-conditioned diffusion model on real 100-asset windows.
2. Generates synthetic windows and gates them with the 9-point stylized-fact check.
3. Packs accepted windows into WebDataset tars and uploads them to HuggingFace.

Downstream notebooks (`colab_pretrain.ipynb`, `colab_portfolio_return.ipynb`,
`colab_portfolio_sharpe.ipynb`) download and unpack these tars automatically.

**Run this before the other training notebooks.**

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────
HF_USERNAME   = 'Verdenroz'
HF_TOKEN      = ''            # HF token with WRITE access (required for upload)
                              # create at: huggingface.co/settings/tokens
HF_MODEL_REPO = f'{HF_USERNAME}/meridian-checkpoints'
HF_DATA_REPO  = f'{HF_USERNAME}/meridian-data'
FDA_TAR_REPO  = f'{HF_USERNAME}/meridian-fda-synth'
FDA_TAR_PRIVATE = True        # set False to make the synth dataset public

# Spec §11.4c target. Reduce to 5_000–10_000 for a quick end-to-end smoke test.
TARGET_VALID_WINDOWS = 50_000

In [ ]:
# ── 1. Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/meridian'
os.makedirs(f'{DRIVE_DIR}/data/mmaps', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data/fda',   exist_ok=True)
print('Drive ready:', DRIVE_DIR)

In [ ]:
# ── 2. HuggingFace login + install deps ───────────────────────────────────────────────
import subprocess, sys
subprocess.run([
    'pip', 'install', '-q',
    'numpy>=2.1', 'pandas>=2.2', 'pyyaml>=6.0',
    'hmmlearn>=0.3', 'faiss-cpu>=1.8', 'tokenizers>=0.20',
    'huggingface_hub>=0.26',
], check=True)

from huggingface_hub import login, snapshot_download, hf_hub_download, HfApi
if HF_TOKEN:
    login(token=HF_TOKEN)
print('HuggingFace ready.')

In [ ]:
# ── 3. Download training code + symbols file ──────────────────────────────────────────
import os, shutil

if not os.path.exists('/content/meridian'):
    local = snapshot_download(repo_id=HF_MODEL_REPO, allow_patterns='code/**')
    shutil.copytree(os.path.join(local, 'code'), '/content/meridian')
    print('Code downloaded.')
else:
    print('Code already present.')

symbols_path = '/content/meridian/symbols.txt'
if not os.path.exists(symbols_path):
    path = hf_hub_download(repo_id=HF_MODEL_REPO, filename='symbols.txt')
    shutil.copy(path, symbols_path)
    print(f'symbols.txt downloaded → {symbols_path}')
else:
    print('symbols.txt already present.')

In [ ]:
# ── 4. Download real shards (cached to Drive across sessions) ───────────────────────────
import os, shutil, time

def _drive_retry(fn, what, attempts=6, base_sleep=3):
    last = None
    for i in range(attempts):
        try:
            return fn()
        except OSError as e:
            last = e
            print(f'[{what}] Drive EIO (attempt {i + 1}/{attempts}): {e}; sleeping {(i + 1) * base_sleep}s')
            time.sleep((i + 1) * base_sleep)
    raise RuntimeError(
        f'{what} failed after {attempts} retries. '
        f'Remount with: from google.colab import drive; '
        f'drive.flush_and_unmount(); drive.mount("/content/drive", force_remount=True)'
    ) from last

data_dir  = f'{DRIVE_DIR}/data/mmaps'
train_dir = f'{data_dir}/train'

def _has_data():
    if not os.path.exists(train_dir):
        return False
    entries = _drive_retry(lambda: os.listdir(train_dir), 'list train_dir')
    return any(f.endswith('.f32bin') for f in entries)

if not _has_data():
    snapshot_download(repo_id=HF_DATA_REPO, repo_type='dataset', local_dir=data_dir)
    for prefix in [os.path.join(data_dir, 'data', 'mmaps'), os.path.join(data_dir, 'mmaps')]:
        if os.path.exists(os.path.join(prefix, 'train')):
            for item in os.listdir(prefix):
                src = os.path.join(prefix, item)
                dst = os.path.join(data_dir, item)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(
                os.path.join(data_dir, prefix.replace(data_dir + '/', '').split('/')[0]),
                ignore_errors=True,
            )
            break
    print(f'Data downloaded → {data_dir}')
else:
    print('Data already in Drive.')

In [ ]:
# ── 5. Stage real shards on local SSD ───────────────────────────────────────────────
# FDA only needs the train split. FDA_SYNTH_* files are not mirrored to Drive
# (step 8 pushes them straight to HF as packed tars).
import os, shutil

local_data  = '/content/data/mmaps'
local_train = f'{local_data}/train'
os.makedirs(local_train, exist_ok=True)

drive_train = f'{DRIVE_DIR}/data/mmaps/train'
real_files  = [f for f in os.listdir(drive_train)
               if f.endswith('.f32bin') and not f.startswith('FDA_SYNTH_')]
print(f'Copying {len(real_files)} real shards from Drive to local SSD...')
for f in real_files:
    dst = os.path.join(local_train, f)
    if not os.path.exists(dst):
        shutil.copy2(os.path.join(drive_train, f), dst)
print(f'Done. Real train shards on SSD: {len(real_files)}')

In [ ]:
# ── 6. Auto-detect GPU ─────────────────────────────────────────────────────────────
import torch, os
assert torch.cuda.is_available(), 'No GPU — enable it in Runtime → Change runtime type.'

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)}  ({vram_gb:.0f} GB VRAM)')

# FDA diffusion model is lightweight (~1–2 GB VRAM). Batch size mainly affects
# training speed; max_real_windows caps how many real windows the diffusion
# model trains on (higher = better coverage of the distribution).
if vram_gb >= 35:
    FDA_BATCH_SIZE       = 256
    FDA_GEN_BATCH_SIZE   = 256
    FDA_MAX_REAL_WINDOWS = 50_000
elif vram_gb >= 22:
    FDA_BATCH_SIZE       = 128
    FDA_GEN_BATCH_SIZE   = 128
    FDA_MAX_REAL_WINDOWS = 30_000
else:
    FDA_BATCH_SIZE       = 64
    FDA_GEN_BATCH_SIZE   = 64
    FDA_MAX_REAL_WINDOWS = 20_000

print(f'fda_batch={FDA_BATCH_SIZE}  gen_batch={FDA_GEN_BATCH_SIZE}  max_real_windows={FDA_MAX_REAL_WINDOWS}')

In [ ]:
# ── 7. Run FDA pipeline ─────────────────────────────────────────────────────────────
import os, sys, subprocess

OUTPUT_DIR = '/content/data/mmaps'

cmd = [
    sys.executable, '-u',
    '-m', 'train.fda_pipeline',
    '--input-dir',             f'{OUTPUT_DIR}/train',
    '--output-dir',            OUTPUT_DIR,
    '--max-real-windows',      str(FDA_MAX_REAL_WINDOWS),
    # window-len = seq_len (64) + positive_offset (4) so each synth shard
    # produces a valid TCP pair (W_t, W_{t+4}). Values < 68 silently exclude
    # synth shards from TCP contrastive training (spec §11.3).
    '--window-len',            '68',
    '--stride',                '4',
    '--epochs',                '200',
    '--batch-size',            str(FDA_BATCH_SIZE),
    '--generation-batch-size', str(FDA_GEN_BATCH_SIZE),
    '--diffusion-steps',       '1000',   # spec §8.2
    '--target-valid-windows',  str(TARGET_VALID_WINDOWS),
    '--max-generation-rounds', '5000',
    '--max-abs-feature',       '10.0',
    '--soft-target-pass',      '0.90',
    '--eta',                   '0.0',    # DDIM (deterministic, best moment match)
    '--x0-clip',               '5.0',
    '--device',                'cuda',
]

env = os.environ.copy()
env['PYTHONPATH']       = '/content/meridian'
env['PYTHONUNBUFFERED'] = '1'

print('Running:', ' '.join(cmd))
print('-' * 80)

process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'FDA pipeline failed with return code {process.returncode}')
print('FDA generation complete.')

In [ ]:
# ── 8. Pack FDA shards into tars and push to HuggingFace ──────────────────────────
# Packs FDA_SYNTH_*.f32bin files into ~100 MiB WebDataset tars and uploads
# them to FDA_TAR_REPO. Downstream notebooks stream these tars directly from
# HF and unpack into their local SSD — no Drive copy of synth data needed.
import os, sys, subprocess, shutil

SRC_TRAIN   = '/content/data/mmaps/train'
TAR_OUT_DIR = '/content/data/fda_tars'

synth_files = [f for f in os.listdir(SRC_TRAIN) if f.startswith('FDA_SYNTH_')]
if not synth_files:
    raise RuntimeError('No FDA_SYNTH_* shards found in local train dir — did step 7 complete?')
print(f'{len(synth_files)} FDA synth shards to pack.')

if not HF_TOKEN:
    raise RuntimeError(
        'HF_TOKEN is empty — set it in the Configuration cell. '
        'Requires WRITE access at huggingface.co/settings/tokens.'
    )

cmd = [
    sys.executable, '-u', '-m', 'tools.pack_fda_shards',
    '--src',             SRC_TRAIN,
    '--out',             TAR_OUT_DIR,
    '--target-mib',      '100',
    '--source-manifest', f'{OUTPUT_DIR}/fda_manifest.json',
    '--upload-repo',     FDA_TAR_REPO,
]
if FDA_TAR_PRIVATE:
    cmd.append('--upload-private')

env = os.environ.copy()
env['PYTHONPATH'] = '/content/meridian'
env['HF_TOKEN']   = HF_TOKEN

process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'pack_fda_shards failed with return code {process.returncode}')

# Mirror the small tar manifest to Drive so it survives session resets.
manifest_src = os.path.join(TAR_OUT_DIR, 'fda_manifest_tars.json')
if os.path.exists(manifest_src):
    shutil.copy2(manifest_src, f'{DRIVE_DIR}/data/fda/fda_manifest_tars.json')
    print('Tar manifest mirrored to Drive.')

print()
print(f'Upload complete → https://huggingface.co/datasets/{FDA_TAR_REPO}')